In [ ]:
import os
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/*

In [ ]:
import glob, os, shutil

MICROGHOST_INPUTS = [
    '/kaggle/input/datasets/hassansara/microghost',
    '/kaggle/input/microghost',
]

def find_microghost_project(base):
    if not os.path.exists(base):
        return None
    if os.path.exists(os.path.join(base, 'main.py')):
        return base
    for candidate in glob.glob(os.path.join(base, '**', 'main.py'), recursive=True):
        root = os.path.dirname(candidate)
        if os.path.exists(os.path.join(root, 'config.py')):
            return root
    return None

src = next((p for p in (find_microghost_project(base) for base in MICROGHOST_INPUTS) if p), None)
if src is None:
    raise FileNotFoundError('Attach the patched MicroGhost Kaggle dataset. Expected one of: ' + ', '.join(MICROGHOST_INPUTS))

shutil.copytree(src, '/kaggle/working/microghost', dirs_exist_ok=True)
print(f'Copied patched MicroGhost project from: {src}')

In [ ]:
%cd /kaggle/working/microghost
print('Using patched MicroGhost code from the Kaggle input dataset. No network code download is used.')

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install onnxscript -q

In [ ]:
import os
import sys

DATASET_PATHS = {
    'MICROGHOST_FORESTPERSONS_PATH': ('forestpersons', [
        '/kaggle/input/datasets/sayjadrahman/forestpersons-subset-5k',
        '/kaggle/input/forestpersons-subset-5k',
    ]),
    'MICROGHOST_FORESTPERSONSIR_PATH': ('forestpersons', [
        '/kaggle/input/datasets/sayjadrahman/forestpersonsir-subset-5k',
        '/kaggle/input/forestpersonsir-subset-5k',
    ]),
    'MICROGHOST_LLVIP_PATH': ('llvip', [
        '/kaggle/input/datasets/afradhossain/llvip-dataset',
        '/kaggle/input/llvip-dataset',
    ]),
}

def find_dataset_root(base, kind):
    if not os.path.exists(base):
        return None
    search_roots = [base] + [p for p in glob.glob(os.path.join(base, '**'), recursive=True) if os.path.isdir(p)]
    for root in search_roots:
        names = {name.lower() for name in os.listdir(root)}
        if kind == 'llvip' and {'annotations', 'infrared', 'visible'}.issubset(names):
            return root
        if kind == 'forestpersons' and ('images' in names or 'annotations' in names or 'annotations.zip' in names):
            has_json = bool(glob.glob(os.path.join(root, '**', '*.json'), recursive=True))
            has_zip = os.path.exists(os.path.join(root, 'annotations.zip'))
            if has_json or has_zip:
                return root
    return None

for env_key, (kind, candidates) in DATASET_PATHS.items():
    path = next((find_dataset_root(p, kind) for p in candidates if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError(f'Missing Kaggle input for {env_key}. Tried: {candidates}')
    os.environ[env_key] = path
    print(f'{env_key} = {path}')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MICROGHOST_MAX_GRAD_NORM'] = '2.0'
sys.path.append('/kaggle/working/microghost')
import config

print(f'DEBUG_MODE is set to: {config.DEBUG_MODE}')
print(f"MAX_GRAD_NORM is set to: {os.environ['MICROGHOST_MAX_GRAD_NORM']}")

In [ ]:
%cd /kaggle/working/microghost
# Set RUN_DEBUG_FIRST=True only when you want a tiny smoke test before full training.
RUN_DEBUG_FIRST = False
BATCH_SIZE = 16
NUM_WORKERS = 2

if RUN_DEBUG_FIRST:
    !python main.py train --debug --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}
else:
    !python main.py train --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

print('\nTraining cell complete.')
